# 05. Эмбеддинги обычного AE и AE с удалением шума

`MODE="embeddings"` показывает результаты полного сравнения. `MODE="pilot"` запускает опыт на одной траектории. Режим задаётся в первой ячейке.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print
from tsprep.evaluation.experiment_results import load_experiment_tables, task_summary
from tsprep.evaluation.visualize import plot_task_metric

def show_task(frame, metrics):
    if frame.empty:
        print("Нет данных для этой задачи. Отсутствующая метрика не заменяется нулём.")
    else:
        display(task_summary(frame, metrics))

MODE = "embeddings"  # "pilot" runs the historical debug experiment
RESULT = ROOT / "results" / "embedding_robustness_v1_20261105"


## Полный эксперимент с эмбеддингами

Укажите в `RESULT` папку с результатами `embedding_robustness_v1`.
Команда обучения из корня проекта: `python -m tsprep.representation_experiment --out results/embedding_robustness_v1_20261105 --device cuda`.
Для нового запуска измените выходную папку в команде и в `RESULT`.


In [ ]:
if MODE == 'embeddings':
    from tsprep.provenance import read_json, file_hash
    from IPython.display import Image
    if not (RESULT / 'complete.json').exists():
        raise RuntimeError('Эксперимент ещё не завершён. Дождитесь complete.json или укажите другой RESULT.')
    completion = read_json(RESULT / 'complete.json')
    if completion['status'] != 'complete':
        raise ValueError('Smoke не является подтверждающим опытом')
    for name, expected in completion['files'].items():
        if file_hash(RESULT / name) != expected:
            raise ValueError(f'Неверная контрольная сумма: {name}')
    selection = read_json(RESULT / 'selection.json')
    decision = read_json(RESULT / 'decision.json')
    protocol = read_json(RESULT / 'protocol.json')
    assert selection['frozen_before_confirmation']
    print('Проверены файлы:', len(completion['files']))
    print('Общая learning rate:', selection['learning_rate'])
    print('Время, секунд:', round(completion['seconds'], 1))
    display(pd.DataFrame([{'arm': r['arm'], 'lr': r['lr'], 'best_epoch': r['history']['best_epoch'],
                          'validation_clean_MSE': r['history']['best_score'],
                          'validation_forecast_ratio_to_raw': r['validation_forecast_ratio']}
                         for r in selection['candidates']]))


### Прогноз по фиксированным эмбеддингам

Все методы прогнозируют одно чистое стандартизованное значение Δlog P точно через H=7.
Энкодер обучается восстанавливать окно. После фиксации его весов обучается Ridge.
Обычный AE и AE с удалением шума имеют одинаковый размер z=8 и общую скорость обучения.
`clean_oracle` использует чистый вход для диагностики; на рынке такой эталон недоступен.


In [ ]:
if MODE == 'embeddings':
    summary = pd.read_csv(RESULT / 'summary.csv')
    display(summary[summary.severity == 1].sort_values('mae'))
    metrics = pd.read_csv(RESULT / 'forecast_metrics.csv')
    trajectory = metrics.groupby(['severity', 'family', 'data_seed', 'model'])['mae'].mean().unstack('model')
    relative = trajectory.div(trajectory['raw'], axis=0)
    display(relative.groupby('severity').mean())
    display(Image(filename=str(RESULT / 'embedding_quality.png')))
    display(pd.read_csv(RESULT / 'summary_by_family.csv').query('severity == 1'))


### Основная гипотеза и ограничения

AE с удалением шума сравнивается с обычным AE, PCA-8 и global MAD→PCA-8 при стандартном загрязнении.
Критерий выполнен, если все три верхние односторонние границы ниже 1; квантиль 0.98333… учитывает поправку Bonferroni.
Бутстреп повторно выбирает независимые траектории внутри семейств. Ошибки трёх инициализаций усредняются внутри траектории; прогнозы не объединяются в ансамбль.
Отсутствие загрязнений и удвоенные ошибки проверяются на тех же траекториях. Это дополнительные условия проверки, а не новые независимые ряды.


In [ ]:
if MODE == 'embeddings':
    display(pd.DataFrame([{'reference': name, **interval} for name, interval in decision['comparisons'].items()]))
    print('Совместный критерий:', decision['joint_success'])
    print('Верхний квантиль:', decision['upper_quantile'])
    reconstruction = pd.read_csv(RESULT / 'reconstruction_metrics.csv')
    display(reconstruction.query('severity == 1').groupby(['family', 'data_seed', 'model'])[['all_mae', 'normal_mse']].mean().groupby('model').mean())
    latent = pd.read_csv(RESULT / 'latent_metrics.csv')
    display(latent.query('severity == 1').groupby(['family', 'data_seed', 'model'])[['dimension', 'effective_rank', 'active_units']].mean().groupby('model').mean())
    print('Variance/rank — диагностические показатели, а не количество информации.')


In [ ]:
if MODE == 'embeddings':
    family = 'seasonal_period_11'
    data_seed = protocol['confirmation_seeds'][0]
    with np.load(RESULT / 'predictions' / f'{family}_{data_seed}_severity1.npz') as saved:
        target_time = saved['endpoint_index'][:150] + protocol['forecast_horizon']
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(target_time, saved['target'][:150], label='clean H=7 target', alpha=.7)
        for name in ['raw', 'pca', 'mad_pca', 'denoising_seed111']:
            ax.plot(target_time, saved['prediction_'+name][:150], label=name+' + Ridge')
        ax.set(xlabel='Target time index', ylabel='Normalized log return', title='Fixed first seasonal confirmation trajectory')
        ax.legend()
        plt.tight_layout()
        plt.show()


## Пилотный опыт: TCN-AE на одной траектории

Ячейки выполняются при `MODE = 'pilot'`. Параметр генерации данных 9001, обучение с ранней остановкой.


In [ ]:
if MODE == "pilot":
    from datetime import datetime, timezone
    from tsprep.config import DataConfig, ModelConfig, TrainConfig, ExperimentConfig
    from tsprep.data_preprocessing.synthetic import generate_series
    from tsprep.data_preprocessing.datasets import prepare_and_window
    from tsprep.experiment import train_experiment, evaluate_baselines, load_artifact
    from tsprep.evaluation.examples import downstream_example

    DEVICE = "cpu"
    EPOCHS = 40
    DATA_SEED, MODEL_SEED, LOADER_SEED = 9001, 317, 317
    PILOT_DIR = ROOT/"results"/datetime.now(timezone.utc).strftime("denoising_pilot_%Y%m%dT%H%M%S%fZ")
    sample = generate_series(length=2400, seed=DATA_SEED)
    data_cfg = DataConfig(window_len=24, stride=2, batch_size=128, forecast_horizon=7, seed=LOADER_SEED)
    model_cfg = ModelConfig(model_name="tcn_ae", hidden_dim=16, latent_dim=8, tcn_levels=2, dropout=0)
    prepared = prepare_and_window(sample.values, data_cfg, is_anomaly=sample.is_anomaly)
    baselines = evaluate_baselines(prepared, data_cfg, latent_dim=model_cfg.latent_dim, clean_raw=sample.clean_values)
    print(PILOT_DIR, DEVICE, "parameters and seeds fixed before test")

## Две цели обучения
Обе модели получают только наблюдаемые значения. Нормализатор обучается на наблюдаемой обучающей выборке и применяется также к чистому эталону. Чистая проверочная выборка используется обеими моделями только для выбора весов. Тестовые данные в выборе не участвуют; чистый тестовый эталон нужен только для оценки.


In [ ]:
if MODE == "pilot":
    pilot_metrics, pilot_examples = {}, {}
    for arm, target in (("ordinary", "observed"), ("denoising", "clean")):
        experiment = ExperimentConfig(data_cfg, model_cfg,
            TrainConfig(epochs=EPOCHS, seed=MODEL_SEED, device=DEVICE, deterministic=True,
                training_target=target, validation_target="clean", selection_metric="endpoint_mae",
                early_stopping_patience=10))
        path = PILOT_DIR/arm
        pilot_metrics[arm] = train_experiment(experiment, sample.values, output_dir=path,
            clean_raw=sample.clean_values, is_anomaly=sample.is_anomaly,
            metadata={"purpose": "debug_pilot", "data_seed": DATA_SEED, "training_target": target}, verbose=False)
        pilot_examples[arm] = downstream_example(path, device=DEVICE, baselines=baselines)
        print(arm, "epochs:", pilot_metrics[arm]["epochs_ran"], "best:", pilot_metrics[arm]["best_epoch"],
            "validation endpoint clean MAE:", pilot_metrics[arm]["validation_score"])

In [ ]:
if MODE == "pilot":
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for arm in ("ordinary", "denoising"):
        _, _, _, manifest, _ = load_artifact(PILOT_DIR/arm)
        h = manifest["history"]
        axes[0].plot(np.arange(1, len(h["train_recon"])+1), h["train_recon"], label=arm)
        axes[1].plot(np.arange(1, len(h["selection_loss"])+1), h["selection_loss"], label=arm)
    axes[0].set(xlabel="Epoch", ylabel="Training MSE", title="Training targets differ between arms")
    axes[1].set(xlabel="Epoch", ylabel="Validation endpoint clean MAE", title="Same checkpoint criterion for both arms")
    for axis in axes:
        axis.legend()
    fig.tight_layout()
    plt.show()

## Простые методы сравнения и раздельные ошибки
PCA той же размерности обучается только на наблюдаемой обучающей выборке. Длина медианного фильтра выбирается из 3/5/9 по MAE последней точки к чистому эталону на проверочной выборке. Фильтр использует только прошлое и текущую точку. Постоянный метод выдаёт среднее наблюдаемой обучающей выборки. Для каждого метода показаны все, обычные и загрязнённые концы окон. Используются одна генерация данных и одна инициализация, поэтому результаты описательные.


In [ ]:
if MODE == "pilot":
    from tsprep.data_preprocessing.datasets import prepare_and_window
    clean_prepared = prepare_and_window(sample.clean_values, data_cfg, normalizer=prepared.normalizer)
    reference = pilot_examples["ordinary"]
    endpoint = reference["endpoint"]
    normal = ~reference["anomaly"]["endpoint_labels"].astype(bool)
    validation_clean = clean_prepared.val_windows[:, -1]
    median_length = min((3, 5, 9), key=lambda k: np.abs(np.median(prepared.val_windows[:, -k:], axis=1)-validation_clean).mean())
    methods = dict(observed_input=endpoint["observed"], PCA=endpoint["PCA"],
        train_mean=np.broadcast_to(prepared.raw_normalized[:prepared.split_bounds["train"][1]].mean(axis=0), endpoint["clean"].shape),
        causal_median=np.median(prepared.test_windows[:, -median_length:], axis=1),
        ordinary=pilot_examples["ordinary"]["endpoint"]["reconstructed"],
        denoising=pilot_examples["denoising"]["endpoint"]["reconstructed"])
    pilot_rows = []
    for method, values in methods.items():
        for subset, mask in (("all", np.ones(len(normal), bool)), ("normal", normal), ("contaminated", ~normal)):
            error = (values-endpoint["clean"])[mask]
            pilot_rows.append(dict(method=method, subset=subset, n=int(mask.sum()),
                mse=float(np.square(error).mean()) if mask.any() else None,
                mae=float(np.abs(error).mean()) if mask.any() else None))
    pilot_table = pd.DataFrame(pilot_rows)
    display(pilot_table)
    print("Validation-selected causal median length:", median_length)
    observed_all = pilot_table[(pilot_table.method == "observed_input") & (pilot_table.subset == "all")].iloc[0]
    observed_normal = pilot_table[(pilot_table.method == "observed_input") & (pilot_table.subset == "normal")].iloc[0]
    for arm in ("ordinary", "denoising"):
        overall = pilot_table[(pilot_table.method == arm) & (pilot_table.subset == "all")].iloc[0]
        ordinary_points = pilot_table[(pilot_table.method == arm) & (pilot_table.subset == "normal")].iloc[0]
        print(arm, "clean MAE / input:", overall.mae/observed_all.mae,
              "normal MSE / input:", ordinary_points.mse/observed_normal.mse)

In [ ]:
if MODE == "pilot":
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for axis, metric in zip(axes, ("mse", "mae")):
        subset = pilot_table[pilot_table.subset == "all"]
        axis.barh(subset.method, subset[metric])
        axis.set_title("Clean endpoint "+metric.upper()+" (absolute, nonnegative)")
    fig.tight_layout()
    plt.show()
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(endpoint["time"][:160], endpoint["clean"][:160, 0], label="clean", color="black")
    for method in ("observed_input", "ordinary", "denoising", "causal_median"):
        ax.plot(endpoint["time"][:160], methods[method][:160, 0], label=method, alpha=.7)
    ax.legend()
    ax.set_title("Same test endpoints and scale for every method")
    plt.show()

## Отдельная проверка прогноза
Цель прогноза задаётся наблюдаемым преобразованным значением ровно в t+7. Ошибки удаления загрязнений и прогноза оцениваются отдельно. Сравниваются z->Ridge, reconstructed window->Ridge, raw-window Ridge и PCA+Ridge; коэффициент регуляризации выбирается только на проверочной выборке. Эти результаты не используются для повторного выбора модели удаления шума.


In [ ]:
if MODE == "pilot":
    forecast_rows = []
    for arm, example in pilot_examples.items():
        f = example["forecast"]
        for method, prediction in f["predictions"].items():
            forecast_rows.append(dict(arm=arm, method=method, mae=float(np.abs(f["target"]-prediction).mean())))
    display(pd.DataFrame(forecast_rows))
    print("Это отладочный опыт. Для подтверждения нужны независимые траектории, несколько initialization seeds и trajectory-level intervals.")